In [52]:
import pandas as pd
import numpy as np

In [53]:
esg_tickers = pd.read_csv("../Refinitiv ESG Final Data for Analysis.csv")["Symbol"]


ticker_metadata = pd.read_csv("SP500_ticker_metadata.csv")

tickers = ticker_metadata[ticker_metadata['days_since_ipo'] > 180]['ticker']
# use mask to filter out tickers found in esg dataset that do not exist
# and dual classes of stock
# remove PEAK,
# PXD (pioneer energy, aquired by exxon mobil),
# WRK (sidney australia listing)
# CDAY renamed to DAY
# FLT listed in australia
# GOOG (class C to GOOGL)
# FOX (class B to FOXA)
# NWS (class B to NWSA)

tickers = tickers[~tickers.isin(["PEAK", "PXD", "WRK", "CDAY", "FLT", "GOOG", "FOX", "NWS"])]

# intersection of tickers in ESG data and the SP500 data we have
# todo: only get the timeseries data for the ESG tickers we have
tickers = pd.Series(list(set(esg_tickers) & set(tickers))).sort_values(ignore_index=True)


In [54]:
timeseries_13_18 = pd.read_csv("SP500_timeseries_1-1-13--12-31-18.csv")
timeseries_19_24 = pd.read_csv("SP500_timeseries_1-1-19--11-1-24.csv")

# remove columns (days) with no data, for example holidays
timeseries_13_18 = timeseries_13_18.dropna(axis=1, how='all')
timeseries_19_24 = timeseries_19_24.dropna(axis=1, how='all')

# transpose so each column turns in a timeseries for one ticker
# also reverse order so dates increase from top to bottom
timeseries_13_18 = timeseries_13_18.set_index('ticker').T.iloc[::-1]
timeseries_13_18.columns.name = None
timeseries_19_24 = timeseries_19_24.set_index('ticker').T.iloc[::-1]
timeseries_19_24.columns.name = None

# Concatenate along the rows (axis=0)
timeseries = pd.concat([timeseries_13_18.reset_index(), timeseries_19_24.reset_index()],
                       ignore_index=True, sort=False)

timeseries.rename(columns={timeseries.columns[0]: 'date'}, inplace=True)
timeseries.set_index('date', inplace=True)

# only keep tickers we're interested in
timeseries = timeseries.filter(items=tickers)

In [55]:
# replace each entry with the log return compared to the previous day
# first row will turn into NaN so remove it
timeseries = (timeseries/timeseries.shift(1)).iloc[1:].map(np.log)

In [56]:
# calculate covariance of all tickers
# use cov_matrix to calculate correlation between all tickers
# remove tickers that are too highly correlated

# for every ticker, figure out the first day present in the dataset
# then cov(A, B) and corr(A,B) only use the data that is present for both A and B

first_valid_dates = [timeseries[ticker].first_valid_index()
                       for ticker in tickers]

def covariance(x, y):
    x_bar = np.mean(x)
    y_bar = np.mean(y)
    return np.sum((x - x_bar) * (y - y_bar)) / (len(x) - 1)

# calculate covariance of all tickers
cov_matrix = pd.DataFrame(np.nan, index=tickers, columns=tickers)

for i in range(len(tickers)):
    print(f"calculating covariances for {tickers[i]}         ", end="\r", flush=True)
    for j in range(i, len(tickers)):  # Loop over upper triangle

        # start at the later date after which both tickers have data
        start_date = max(first_valid_dates[i], first_valid_dates[j])
        
        ticker_i_slice = timeseries.loc[timeseries.index > start_date, tickers[i]]
        ticker_j_slice = timeseries.loc[timeseries.index > start_date, tickers[j]]
        
        cov_matrix.iloc[i, j] = cov_matrix.iloc[j, i] = covariance(ticker_i_slice, ticker_j_slice)


In [57]:
# use cov_matrix to calculate corr_matrix

std_devs = np.sqrt(np.diagonal(cov_matrix.values))
# Create a matrix of standard deviations (outer product of std_devs with itself)
# Compute correlation matrix
corr_matrix = cov_matrix / np.outer(std_devs, std_devs)
'''
# Apply upper triangle mask to get the upper triangle values
# k=1 to exclude the diagonal
upper_triangle = (corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1) == 1)).stack()

high_correlations = [(index[0], index[1], value) 
                     for index, value in upper_triangle.items() 
                     if value > 0.95]

print(high_correlations)

# the highly correlated tickers are mostly different classes of the same ticker
# [('FRT', 'REG', 0.9118897518213701), ('MET', 'PRU', 0.9005313958213659)]
# significantly different enough over 10 years to leave both in.
'''

"\n# Apply upper triangle mask to get the upper triangle values\n# k=1 to exclude the diagonal\nupper_triangle = (corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1) == 1)).stack()\n\nhigh_correlations = [(index[0], index[1], value) \n                     for index, value in upper_triangle.items() \n                     if value > 0.95]\n\nprint(high_correlations)\n\n# the highly correlated tickers are mostly different classes of the same ticker\n# [('FRT', 'REG', 0.9118897518213701), ('MET', 'PRU', 0.9005313958213659)]\n# significantly different enough over 10 years to leave both in.\n"

In [58]:
n = len(tickers)

# diagonal matrix of volatilities
vol_matrix = np.diag(np.diag(cov_matrix.values))

# matrix of average correlations, except for diagonal of 1s
rho = np.mean(corr_matrix)
average_corr_matrix = np.full((n, n), rho)
np.fill_diagonal(average_corr_matrix, 1)

# F = V C V
structured_cov_matrix = vol_matrix @ average_corr_matrix @ vol_matrix

# print(f"vol_matrix positive semi definite: {np.all(np.linalg.eigvals(vol_matrix) > 0)}")
# print(f"average_corr_matrix positive semi definite: {np.all(np.linalg.eigvals(average_corr_matrix) > 0)}")
# print(f"structured_cov_matrix positive semi definite: {np.all(np.linalg.eigvals(structured_cov_matrix) > 0)}")


print("Condition number of sample covariance matrix:", np.linalg.cond(cov_matrix))
print("Condition number of structured covariance matrix:", np.linalg.cond(structured_cov_matrix))

delta = 0.75

ledoit_wolf_cov_matrix = (1-delta) * cov_matrix + delta * structured_cov_matrix

eigenvalues, eigenvectors = np.linalg.eigh(ledoit_wolf_cov_matrix)
eigenvalues = np.clip(eigenvalues, 1e-5, None)  # Clip eigenvalues smaller than epsilon
ledoit_wolf_cov_matrix = eigenvectors @ np.diag(eigenvalues) @ eigenvectors.T

print(f"determinant {np.linalg.det(ledoit_wolf_cov_matrix)}")

is_positive_semidefinite = np.all(eigenvalues >= 0)

print("Ledoit Wolf cov matrix positive semidefinite?", is_positive_semidefinite)

print("Condition number of ledoit wolf covariance matrix:", np.linalg.cond(ledoit_wolf_cov_matrix))

Condition number of sample covariance matrix: 52149.206789898344
Condition number of structured covariance matrix: 4377.21570812041
determinant 0.0
Ledoit Wolf cov matrix positive semidefinite? True
Condition number of ledoit wolf covariance matrix: 1573.6259912272492


In [59]:
adjusted_cov_matrix_df = pd.DataFrame(ledoit_wolf_cov_matrix, columns=tickers, index=tickers)

adjusted_cov_matrix_df.to_csv("sp500_adjusted_cov_matrix.csv", index=True)


# print(timeseries.shape)
# Save the combined DataFrame to CSV
timeseries.to_csv("sp500_timeseries_13-24.csv", index=True)
